# Softmax and Cross-Entropy

In Lab 7, `MLPClassifier` read handwritten digits and, for each image, printed **ten** probabilities, one per digit. Every network we've built by hand had **one** output: the probability of class 1. Today we answer two questions about the ten-output case:

1. How does a network turn ten scores into ten probabilities? (**Softmax.**)
2. How do we score ten probabilities against the right answer? (**Cross-entropy**, which turns out to be an old friend.)

To answer the second question properly we'll borrow a few ideas from **information theory**: surprise, Shannon entropy, cross-entropy, and KL divergence. They're worth knowing on their own: every large language model is trained with exactly the loss we build today.

### By the end, you'll be able to

1. Turn $K$ scores into $K$ probabilities with softmax, and explain why the sigmoid is softmax with two classes.
2. Compute the surprise of an outcome in bits, and explain why rare outcomes are more surprising and why independent surprises add.
3. Compute the Shannon entropy of a small distribution, read it as the average number of yes/no questions (bits) the best code needs, and say which distributions have the most and the least.
4. Explain cross-entropy as your average surprise when the data come from $p$ but you believe $q$, and KL divergence as the extra surprise you pay for believing $q$. Give two reasons cross-entropy beats squared error as a training loss for probabilities.
5. Explain why, for a labeled data point, cross-entropy is $-\ln$ (the probability the model gave the true class), and why averaging it over the training data (or a minibatch) estimates the cross-entropy we can't compute directly. Recognize $\ln K$ ($\ln 10 \approx 2.30$ for 10 classes) as the loss of a model that knows nothing.
6. *(Optional, Section 6.)* Show that last week's log-loss is the two-class case of cross-entropy, and that softmax with cross-entropy gives the output error signal $\mathbf{p} - \mathbf{y}$.

> **How to read the code in this notebook.** There are two kinds of code cells.
>
> - **Code worth reading.** Cells that use standard libraries, such as numpy (`np.`), do everyday tasks you'll do yourself. Each is introduced with **Code worth reading** in bold. Read these line by line.
> - **Demo cells.** Cells that import from `ch_6_and_10_demos_helpers_2` draw a plot or a widget. That file (and `ch_6_and_10_demos_helpers.py`, which it uses) was written to keep the notebook on ideas, so you never need to open it.

## Setup

In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)

<a id="output-layer"></a>

## The output layer of a multi-class network

Every network we've built so far ended in **one** output unit. A network that reads digits ends in **ten**, one per digit. Each output unit computes a **score** $z_k$ the same way every unit does: a weighted sum of the last hidden layer's outputs, plus a bias. Then one more step turns the ten scores into ten probabilities:

In [ ]:
from ch_6_and_10_demos_helpers_2 import plot_output_layer

plot_output_layer();

**What the last step has to fix.** The ten scores are just numbers. To be probabilities, they need three things:

1. **Each between 0 and 1.** Scores can be any number: negative, or bigger than 1.
2. **Together, summing to 1.** The image is exactly one of the ten digits, so the probabilities have to split 100% among them.
3. **Bigger score, bigger probability,** with no limit on how confident the network can be when the evidence is strong.

Last week's sigmoid fixes problem 1, but not problem 2. Applied to each output unit separately, each sigmoid answers its own yes/no question ("is it a 3?", "is it an 8?"), and nothing stops the network from saying 0.9 to both. **Softmax**, next, fixes all three at once. It *replaces* the sigmoid as the output layer's activation; the hidden layers keep theirs. `MLPClassifier` does this automatically when there are more than two classes: check `out_activation_` on your Lab 7 model.

<details>
<summary>Optional: why not something simpler?</summary>

- **Divide the raw scores by their total?** Scores can be negative, so some "probabilities" would be negative, and the total could even be 0.
- **A sigmoid on each score, then softmax?** The sigmoid squeezes every score into $(0, 1)$, so the largest possible ratio between two classes' probabilities becomes $e^1 / e^0 = e \approx 2.7$. With 10 classes, the most confident the network could ever be is $e / (e + 9) \approx 0.23$, however clear the image. That breaks problem 3. Softmax needs the raw scores.
- **Why $e$, specifically?** With two classes, softmax turns out to be exactly the sigmoid (the optional box at the end of Section 1), and paired with the cross-entropy loss it gives backpropagation the simplest possible starting point (Section 6).
- **When are separate sigmoids right?** When the labels *aren't* exclusive: a photo can contain a cat *and* a dog. Then each output really is its own yes/no question, and scikit-learn uses a sigmoid on each.

</details>

<a id="section-1"></a>

## 1 · Ten probabilities from ten scores: softmax

In Lab 7, `predict_proba` gave one row of 10 numbers per image, and each row summed to 1. The lab said today would show where they come from. Here's how.

The output layer's $K$ scores, $z_1, \dots, z_K$ ([diagram above](#output-layer)), become $K$ probabilities through **softmax**:

$$
p_k = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}} \qquad \text{for } k = 1, \dots, K
$$

In words, two steps:

1. **Make every score positive** by raising $e$ to it. $e^{z}$ is always positive, and a bigger score gives a bigger $e^{z}$.
2. **Divide by the total,** so the $K$ numbers sum to 1.

The biggest score gets the biggest probability, but every class gets *some* probability.

**Demo S1.** Move the three scores and watch both steps.

In [ ]:
from ch_6_and_10_demos_helpers_2 import softmax_demo

softmax_demo()

**Ask:** Add 5 to every score. What happens to the probabilities? Try it with the sliders after you guess.


<details>
<summary>Answer</summary>

Nothing. Each $e^{z_k}$ is multiplied by the same factor $e^5$, and that factor cancels between the top and the bottom:

$$
\begin{aligned}
\frac{e^{z_k + 5}}{\sum_j e^{z_j + 5}} &= \frac{e^{5}\, e^{z_k}}{e^{5} \sum_j e^{z_j}} \\
&= \frac{e^{z_k}}{\sum_j e^{z_j}}
\end{aligned}
$$

Only the *differences* between the scores matter.

</details>

**Code worth reading.** Softmax for a whole array of scores at once, shape `(n, K)`: one row per data point, one column per class. Each row becomes a row of probabilities.

In [ ]:
def softmax(Z):
    "Each row of scores Z, shape (n, K), becomes a row of K probabilities."
    E = np.exp(Z - Z.max(axis=1, keepdims=True))   # step 1 (subtracting the row's max: see below)
    return E / E.sum(axis=1, keepdims=True)        # step 2: divide by the row's total


Z = np.array([[2.0, 1.0, 0.0],                     # two data points, three classes
              [-1.0, 3.0, 0.5]])
P = softmax(Z)
print(P)
print('row sums:', P.sum(axis=1))

**What to notice:**

- **`keepdims=True`** keeps the row totals as a column, shape `(n, 1)`, so dividing `(n, K)` by `(n, 1)` divides each row by its own total (broadcasting, as in last week's bias vectors).
- **Subtracting each row's max** is the Ask above in action: it changes nothing about the probabilities, but it keeps `np.exp` from overflowing when a score is large, such as $e^{1000}$. Real libraries all do this.

<details>
<summary>Optional: the sigmoid is softmax with two classes</summary>

Last week's network had one output, $p = \sigma(z)$, the probability of class 1 ([last week, Section 2](ch_6_and_10_03_training_networks.ipynb#section-2)). Softmax with two classes gives the same thing. Divide the top and bottom by $e^{z_1}$:

$$
\begin{aligned}
p_1 &= \frac{e^{z_1}}{e^{z_1} + e^{z_2}} \\
&= \frac{1}{1 + e^{z_2 - z_1}} \\
&= \frac{1}{1 + e^{-(z_1 - z_2)}} \\
&= \sigma(z_1 - z_2)
\end{aligned}
$$

Only the difference $z_1 - z_2$ matters (the Ask again), so a two-class network needs just one score, $z = z_1 - z_2$, and the sigmoid. ISL calls the $K$-class version the softmax coding (§4.3.5, eq. 4.13, and §10.2, eq. 10.13).

</details>

Now we have ten probabilities. How do we score them against the right answer? Last week's log-loss was built for one probability. To build the ten-class version, and see why it's the right one, we take a short detour through information theory.

<a id="section-2"></a>

## 2 · Surprise

>  This week's first video, [Shannon's Dream, The Limit of Compressibility](https://www.youtube.com/watch?v=l6DKRf-fAAM)  derives Shannon Entropy from first principles. This notebook builds the intuition and the arithmetic, and leaves the derivation to the video.



Before we can score ten probabilities, we need a way to say how bad it is to have given what actually happened a small probability. Information theory calls it **surprise**. If you believed outcome $x$ had probability $q(x)$, and $x$ happens, your surprise is

$$
\text{surprise}(x) = -\log_2 q(x) \quad \text{bits}
$$

(We write $q$ for *your* belief. Section 4 explains why it needs its own letter.)

- **Some values.** A fair coin lands heads: $-\log_2 \tfrac12 = 1$ bit. You draw one particular card from 8: $-\log_2 \tfrac18 = 3$ bits. Something you were certain of happens: $-\log_2 1 = 0$ bits. Something you said was impossible happens: $-\log_2 0 = \infty$.
- **Bits as yes/no questions.** To pin down one of 8 equally likely cards, ask "is it in the first half?" three times: 3 questions, 3 bits. An outcome with probability $1/2^k$ takes $k$ halving questions to find.
- **Why the log.** Two independent events, with probabilities $q_1$ and $q_2$, both happen with probability $q_1 q_2$. The log turns that product into a sum, so their surprises add:

  $$
  \underbrace{-\log_2(q_1 q_2)}_{\text{surprise of both events happening}} = \underbrace{\big(-\log_2 q_1\big) + \big(-\log_2 q_2\big)}_{\text{sum of individual surprises}}
  $$

  That's the same reason [last week's Section 2](ch_6_and_10_03_training_networks.ipynb#section-2) took the log of the likelihood.

In [ ]:
from ch_6_and_10_demos_helpers_2 import plot_surprise_curve

plot_surprise_curve();

**Ask:** You gave an event probability $1/16$, and it happened. How many bits of surprise?


<details>
<summary>Answer</summary>

$$
\begin{aligned}
-\log_2 \tfrac{1}{16} &= \log_2 16 \\
&= 4 \text{ bits}
\end{aligned}
$$

Four halving questions pin down one of 16 equally likely outcomes.

</details>

<a id="section-3"></a>

## 3 · Shannon entropy: the average surprise

Surprise is a property of one outcome. How should we think about combining the surprise of many outcomes?

If outcomes really happen with probabilities $p(x)$, and you know $p$, each outcome $x$ surprises you by $-\log_2 p(x)$, and $x$ happens a fraction $p(x)$ of the time. Your average surprise is the **Shannon entropy** of $p$:

$$
H(p) = -\sum_x p(x)\,\log_2 p(x)
$$

**Ask:** Is Shannon entropy a property of a single outcome, or of something else?


<details>
<summary>Answer</summary>

Of the whole probability **distribution**. It averages the surprise over every outcome, weighted by how often each one happens. Surprise belongs to one outcome; Shannon entropy belongs to the distribution.

</details>

**Example: tomorrow's weather.** Suppose the weather follows

| Weather $x$ | sunny | cloudy | rain | snow |
|---|---|---|---|---|
| $p(x)$ | $1/2$ | $1/4$ | $1/8$ | $1/8$ |
| surprise $-\log_2 p(x)$ | 1 | 2 | 3 | 3 |

Its Shannon entropy, will be:

$$
\begin{aligned}
H(p) &= \tfrac12 \cdot 1 + \tfrac14 \cdot 2 + \tfrac18 \cdot 3 + \tfrac18 \cdot 3 \\
&= 0.5 + 0.5 + 0.375 + 0.375 \\
&= 1.75 \text{ bits}
\end{aligned}
$$

**What 1.75 bits means.** A weather station sends tomorrow's forecast as bits. A fixed-length code needs 2 bits per day for 4 outcomes (`00`, `01`, `10`, `11`). A smarter code gives common outcomes short code words: sunny `0`, cloudy `10`, rain `110`, snow `111`. Its average length is $\tfrac12 \cdot 1 + \tfrac14 \cdot 2 + \tfrac18 \cdot 3 + \tfrac18 \cdot 3 = 1.75$ bits per day, exactly $H(p)$. Each code word's length is that outcome's surprise. (This is a **Huffman code**, which you may have met in a data structures course.) Shannon proved that no code can average fewer than $H(p)$ bits per outcome.

In [ ]:
from ch_6_and_10_demos_helpers_2 import plot_weather_code

plot_weather_code();

**Code worth reading.** Shannon entropy in one line, checked on the weather.

In [ ]:
def entropy(p):
    "Shannon entropy, in bits, of a distribution p (an array that sums to 1)."
    return -np.sum(p * np.log2(p))


p_weather = np.array([1/2, 1/4, 1/8, 1/8])      # sunny, cloudy, rain, snow
print('H(weather):', entropy(p_weather), 'bits')
print('H(uniform over 4):', entropy(np.full(4, 1/4)), 'bits')

**Two facts, shown and not proved:**

- **The uniform distribution has the most entropy.** With 4 outcomes, it's $\log_2 4 = 2$ bits: every day is equally unpredictable, so no code beats 2 bits.
- **A certain (as in $p(x)=1$) outcome has zero entropy.** If it's always sunny, $H = -1 \cdot \log_2 1 = 0$: you never need to send anything. 

(An outcome with $p(x) = 0$ contributes nothing. Strictly, $0 \log 0$ is undefined, because $\log 0$ is, so we define $0 \log 0 = 0$. This matches the limit: $p \log p \to 0$ as $p \to 0$. It also matches the meaning: an outcome that never happens never surprises you.)

**Ask:** Which has more Shannon Entropy: a fair coin's probability distribution, or a coin's that lands heads 90% of the time?


<details>
<summary>Answer</summary>

The fair coin: 1 bit. The 90% coin:

$$
\begin{aligned}
H &= -\big[\,0.9 \log_2 0.9 + 0.1 \log_2 0.1\,\big] \\
&\approx -\big[\,0.9 \cdot (-0.152) + 0.1 \cdot (-3.32)\,\big] \\
&\approx 0.47 \text{ bits}
\end{aligned}
$$

The 90% coin is more predictable, so on average it surprises you less. Check it with `entropy(np.array([0.9, 0.1]))`.

</details>

<a id="section-4"></a>

## 4 · Believing the wrong distribution: cross-entropy and KL divergence

For quick reference, Shannon entropy from [Section 3](#section-3), the average surprise if you believe the truth:

$$
H(p) = -\sum_x p(x)\,\log_2 p(x)
$$

**The wrong assumption, in words first.** Suppose the weather really follows $p$, but your model (loosely, we'll sometimes call this "your beliefs") assumes some other distribution $q$.

- Suppose some weather event $x$ occurs. How often it occurs is set by its true probability $p(x)$, regardless of what you believe.
- You believe the probability of that event is $q(x)$, so your surprise is $-\log_2 q(x)$. If you think $q(x)$ is small, you just saw *what you think* is an unlikely event, so you're very surprised. If you think $q(x)$ is large, your surprise is low. Your surprise does *not* depend on the real probability of $x$, $p(x)$.
- The event's *actual* probability of occurring was $p(x)$.

Repeat this for many days of weather, that is, many values of $x$. What will be *your* average surprise (remember, you think the distribution for $x$ is $q$)? Sum over all possible events: *actual* probabilities, $p(x)$, times *your* corresponding surprise value, which is based on $q(x)$. This is the **cross-entropy**:

$$
H(p, q) = -\sum_x p(x)\,\log_2 q(x)
$$

**Ask:** Based on your intuition, do you expect your average surprise to be higher when your beliefs are wrong, or when they're correct?


<details>
<summary>Answer</summary>

Higher when they're wrong:

$$
H(p, q) \ge H(p), \quad \text{with equality only when } q = p
$$

A wrong belief can make *some* days less surprising: if you overrate sunny days, a sunny day surprises you less. But you pay for it on the days you underrated, and on average you always lose. In the demo below, try to find a $q$ that beats $H(p)$.

</details>

**The gap has a name.** The difference between these two averages, the extra surprise you pay for believing $q$ when the truth is $p$, is the **KL divergence** (Kullback–Leibler divergence):

$$
D_{\mathrm{KL}}(p \,\Vert\, q) = H(p, q) - H(p)
$$

It's written as cross-entropy *minus* Shannon entropy so it is never negative.

**The average is always based on the truth, $p(x)$. The surprise is determined by your beliefs, $q(x)$.**

**Demo S4.** The truth $p$ is the weather from Section 3. Pick a preset for your beliefs $q$, or move the sliders (they're rescaled to sum to 1). The right panel stacks each outcome's piece, $p(x) \times$ surprise: once measuring your surprise with the truth (height $H(p)$), and once with your beliefs (height $H(p, q)$). The hatched strip is the KL gap.

In [ ]:
from ch_6_and_10_demos_helpers_2 import cross_entropy_demo

cross_entropy_demo()

**Interpreting The Average Surprise Plot:** (rightmost plot) The bar heights are their total Shannon Entropy and Cross Entropy respectively. They're sub-divided, showing how much average surprise (or cross entropy) comes from each event. Click the "confident and wrong" preset. 

**Ask:** Why is sunny accounting for such a large portion of the cross-entropy, and snow for almost none?


<details>
<summary>Answer</summary>

This is about the **"confident and wrong"** preset, where your beliefs are $q$ = 0.01 for sunny, cloudy, and rain, and 0.97 for snow.

Each piece of the stack is (how often the outcome really happens) $\times$ (how surprised you are when it does):

$$
p(x) \cdot \big(-\log_2 q(x)\big)
$$

- **Sunny:** it really happens half the time, $p = 0.5$, and you gave it only $q = 0.01$, so each sunny day surprises you by $-\log_2 0.01 \approx 6.64$ bits. Its piece is $0.5 \times 6.64 \approx 3.32$ bits: a big surprise, on many days.
- **Snow:** you expected it, $q = 0.97$, so a snowy day surprises you by only $-\log_2 0.97 \approx 0.04$ bits, and snow happens on just $1/8$ of days. Its piece is $0.125 \times 0.04 \approx 0.005$ bits: a tiny surprise, on few days.
- **Cloudy and rain** surprise you exactly as much as sunny does (you gave each 0.01), but they happen less often, so their pieces are smaller: about 1.66 and 0.83 bits.

The two factors come from different places: **how often** comes from the truth $p$, and **how surprised** comes from your beliefs $q$. Sunny is the outcome where the two disagree most, on the most days.

</details>

**What to notice:**

- **"Uniform"** gives $H(p, q) = 2$ bits, so the KL gap is $2 - 1.75 = 0.25$ bits. Believing every weather is equally likely is like using the fixed-length 2-bit code: it costs a quarter bit per day more than the best code.
- **"Almost right"** costs only 0.03 bits extra. Small mistakes in your beliefs cost little.
- **No setting of $q$ beats $H(p)$.** The KL divergence is never negative, and it's 0 only when $q = p$. This is called **Gibbs' inequality**.
- **The truth's Shannon entropy doesn't depend on your beliefs.** So making the cross-entropy smaller is the same as making the KL divergence smaller: moving your beliefs toward the truth. That's what training a model will do.

<details>
<summary>Optional: why the KL divergence is never negative</summary>

Write the KL divergence as one sum, using natural logs (changing the base only rescales it, so its sign doesn't change):

$$
\begin{aligned}
D_{\mathrm{KL}}(p \,\Vert\, q) &= H(p, q) - H(p) \\
&= -\sum_x p(x) \ln q(x) + \sum_x p(x) \ln p(x) \\
&= -\sum_x p(x) \ln \frac{q(x)}{p(x)}
\end{aligned}
$$

The log's graph lies below its tangent line at $t = 1$: $\ln t \le t - 1$, with equality only at $t = 1$. Apply it with $t = q(x)/p(x)$, summing over the outcomes with $p(x) > 0$:

$$
\begin{aligned}
\sum_x p(x) \ln \frac{q(x)}{p(x)} &\le \sum_x p(x) \left(\frac{q(x)}{p(x)} - 1\right) \\
&= \sum_x q(x) - \sum_x p(x) \\
&\le 1 - 1 \\
&= 0
\end{aligned}
$$

So $D_{\mathrm{KL}}(p \,\Vert\, q) \ge 0$, with equality only when $q(x) = p(x)$ for every $x$.

</details>

**Ask:** Is the KL divergence from $p$ to $q$ the same as from $q$ to $p$? Try it with a fair coin and a coin that lands heads 90% of the time.


<details>
<summary>Answer</summary>

No. Using $D_{\mathrm{KL}}(p \,\Vert\, q) = \sum_x p(x) \log_2 \dfrac{p(x)}{q(x)}$ (the two sums from the optional box, combined):

$$
\begin{aligned}
D_{\mathrm{KL}}(\text{90\% coin} \,\Vert\, \text{fair}) &= 0.9 \log_2 \tfrac{0.9}{0.5} + 0.1 \log_2 \tfrac{0.1}{0.5} \approx 0.53 \text{ bits} \\
D_{\mathrm{KL}}(\text{fair} \,\Vert\, \text{90\% coin}) &= 0.5 \log_2 \tfrac{0.5}{0.9} + 0.5 \log_2 \tfrac{0.5}{0.1} \approx 0.74 \text{ bits}
\end{aligned}
$$

They differ, so the KL divergence isn't a distance in the everyday sense. The order matters: the first argument is the truth, the one the average is taken over.

</details>

**Where this shows up.** Large language models are trained to make the cross-entropy of their next-word predictions small. The KL divergence appears when a model is fine-tuned: a KL penalty keeps the new model from drifting too far from the original.

<a id="section-5"></a>

## 5 · Why cross-entropy, and not squared error?

We could score a model's probabilities with something more familiar. Say the model gave probability $q$ to what actually happened. 

The **squared error** $(1 - q)^2$ is 0 when the model was certain and right (see the optional explanation below for why that works), and it's what we used for regression. So why does every classifier use cross-entropy, $-\ln q$, instead?

<details>
<summary>Optional: why $(1 - q)^2$? Where's the label/prediction?</summary>

A loss has to compare the prediction with the truth, and this one does. The label is hiding in $q$: $q$ is the probability the model gave to *what actually happened*, and the label tells us *which* probability that is.

To see the label, use last week's notation for a two-class network: label $y_i \in \{0, 1\}$, and network output $p_i$, its probability of class 1. (Here $p_i$ is the network's prediction, as last week, not the truth; [the end of this section](#loss-one-point) sorts out the letters.)

| Label | What happened | $q$, the probability given to it | Squared error $(y_i - p_i)^2$ | In terms of $q$ |
|---|---|---|---|---|
| $y_i = 1$ | class 1 | $q = p_i$ | $(1 - p_i)^2$ | $(1 - q)^2$ |
| $y_i = 0$ | class 0 | $q = 1 - p_i$ | $(0 - p_i)^2 = p_i^2$ | $(1 - q)^2$ |

So $(1 - q)^2$ is just $(y_i - p_i)^2$, target minus prediction, in both cases. The 1 in $1 - q$ is the target: the truth puts probability 1 on what happened. In the same way, $-\ln q$ is last week's log-loss, $-\big[\,y_i \ln p_i + (1 - y_i)\ln(1 - p_i)\,\big]$, in both cases; Section 6 shows this step by step. Writing both losses in terms of $q$ lets us compare them on one axis.

</details>

(This section uses natural logs, $\ln$, which measure surprise in **nats** instead of bits. The base of the log only rescales the loss: 1 nat ≈ 1.44 bits; [Section 6](#section-6) says more.)

The figure compares the two. On the right is each loss's slope with respect to the score $z$ behind $q$, $\partial L / \partial z$, for a sigmoid output $q = \sigma(z)$ as in [last week's network](ch_6_and_10_03_training_networks.ipynb#section-5). Gradient descent changes $z$ by $-\eta\, \partial L / \partial z$, so the size of this slope sets how big a step training takes on $z$: informally, how hard the loss "pushes" on $z$.

In [ ]:
from ch_6_and_10_demos_helpers_2 import plot_ce_vs_squared_error

plot_ce_vs_squared_error();

**The two reasons that matter most:**

1. **Squared error barely punishes confident mistakes** (left panel). It can never exceed 1, so giving what happened probability 0.1 or 0.001 costs almost the same: 0.81 vs 0.998. Cross-entropy keeps climbing: 2.30 vs 6.91. Saying "nearly impossible" about something that happens is the worst thing a probability model can do, and cross-entropy treats it that way (the "confident and wrong" Ask in [Section 4](#section-4)).
2. **Squared error stops learning exactly when the model is most wrong** (right panel). For a sigmoid output, squared error's slope is $\partial L / \partial z = 2(q - 1)\,q\,(1 - q)$. The factor $q$ drives it toward 0 when $q$ is tiny (the orange curve returns to 0 at the left edge), so a confidently wrong network hardly moves. Cross-entropy's slope is $\partial L / \partial z = q - 1$, last week's $p - y$ ([Step 4](ch_6_and_10_03_training_networks.ipynb#section-5)): its size is largest when the model is most wrong (the blue line reaches −1 at the left edge). At $q = 0.001$, cross-entropy's slope is about 500 times larger, and so is the gradient descent step it produces. In practice, this is the main reason networks trained with squared error learn slowly or stall.

This isn't the full list but these are the reasons that matter most for networks.

Squared error on probabilities is still a reasonable way to *evaluate* a model's probabilities after training (it's called the **Brier score**). Its problems are as a *training* loss.

**Beyond two classes.** We worked the simplest case, a two-class network with one sigmoid output, to keep the formulas short. The same two arguments hold for a network with $K$ softmax outputs: squared error summed over all $K$ outputs is still capped (it can never exceed 2), and its slopes with respect to the scores still fade toward 0 when the network is confidently wrong. Cross-entropy's slopes stay prediction minus target, $\mathbf{p} - \mathbf{y}$, as [Section 6](#section-6) shows.

**Ask:** A model gives the true class probability 0.02. Which loss is larger, and which has the larger slope $\partial L / \partial z$, so gradient descent takes a bigger step to fix it?


<details>
<summary>Answer</summary>

Cross-entropy, on both counts.

- **The loss:** $-\ln 0.02 \approx 3.9$, against $(1 - 0.02)^2 \approx 0.96$ for squared error.
- **The slope:** $q - 1 = -0.98$ for cross-entropy, against $2(0.02 - 1)(0.02)(0.98) \approx -0.04$ for squared error, about 25 times smaller.

</details>

<a id="loss-one-point"></a>

### Training on samples: the loss for one labeled data point

**What we want.** Training should make the network's beliefs match the real world: 

- we want the **cross-entropy between the truth and the network's "beliefs"** to be small
- gradient descent needs the slopes of that number.

**Problem:** We can't compute it. 
- We never see the data-generating process (DGP) as an entire distribution. 
- We only have samples from the DGP: labeled data points. 

**The fix: average model's surprise over samples.** Cross-entropy is surprise averaged *over the true distribution* (Section 4). So you don't need the truth written down; you only need samples drawn from it. Average your surprise over the samples, and that average estimates the cross-entropy.

**Code worth reading.** Example using the weather. The truth is some historical weather data, used only to generate the weather each day. The estimate itself uses only the days and $q$.

In [ ]:
rng = np.random.default_rng(0)
p = np.array([1/2, 1/4, 1/8, 1/8])     # the truth (sunny, cloudy, rain, snow): used only to generate days
q = np.array([0.1, 0.2, 0.35, 0.35])   # your beliefs: they underrate sunny

days = rng.choice(4, size=365, p=p)    # a year of weather, sampled from the truth
surprise = -np.log2(q[days])           # your surprise on each day, in bits

print('surprise on the first 10 days:', surprise[:10].round(2))
print('average over 365 days:', surprise.mean().round(3))
print('exact H(p, q):', (-(p * np.log2(q)).sum()).round(3))

**Demo S6.** Left: the running average of your surprise, for three different sampled years. Right: the estimates from 2,000 samples of 10 days and 2,000 samples of 100 days.

In [ ]:
from ch_6_and_10_demos_helpers_2 import plot_sampled_cross_entropy

plot_sampled_cross_entropy();

**What to notice:**

- **The more days, the better the estimate.** Each year's average wanders at first, then settles near the exact $H(p, q)$.
- **Small samples are noisier, but aimed at the right value.** Estimates from 10 days scatter about three times as widely as those from 100 days, and both are centered on the exact value.

**Ask:** When we average surprise over sampled days, what are we estimating: the truth $p$, your beliefs $q$, or something else?


<details>
<summary>Answer</summary>

Something else: the cross-entropy $H(p, q)$ itself, a single number. Your beliefs $q$ are known exactly: you look up $q(x)$ for each sampled day. The truth $p$ is never estimated either; the samples simply stand in for it in the one place the formula uses it, the averaging.

</details>

<details>
<summary>Optional: is this the bootstrap?</summary>

It's related, but not the same. Both treat a sample as a stand-in for the population it came from ([Chapter 5](../ch_5/ch_5_demos_2_bootstrap.ipynb)). The difference is what they estimate:

- **Here, the sample estimates a value:** the cross-entropy, by plain averaging. The usual name for this is a **Monte Carlo estimate**: an average over random samples standing in for an average over a distribution.
- **The bootstrap estimates how much a value varies:** it resamples the data, with replacement, many times, recomputes the statistic each time, and looks at the spread (for example, a standard error).

The right panel above asks the bootstrap's kind of question (how much does the estimate vary from sample to sample?), but it answers it by drawing fresh samples from the truth, which we can only do because we made the truth up. With real data, the bootstrap would approximate that spread by resampling.

**Are we estimating the truth's distribution?** Not as a separate step. One exact way to say it: the average over $n$ samples *is* the cross-entropy between the sample itself (each point given probability $1/n$, called the **empirical distribution**) and your beliefs. So the sample is plugged in where the truth goes, and the number that comes out estimates the true cross-entropy.

</details>

**Back to the network.** When we train a network, the truth for one data point is its **label**, and the beliefs are the **network's prediction**. So from here on, we switch back to the usual machine-learning letters from Lab 7 and last week: $\mathbf{p}$ is the network's predicted probabilities, and $\mathbf{y}$ is the label.

**One labeled data point is one sample of the truth.** Like one day of weather, it tells us one outcome: the class we saw. As a distribution, that one outcome puts probability 1 on the observed class and 0 on every other class. As a vector, that's a **one-hot** vector: with 3 classes and the third one observed, $\mathbf{y} = (0, 0, 1)$.

**Only one term of the cross-entropy survives.** Say the network predicts $\mathbf{p} = (0.2, 0.5, 0.3)$. In natural logs:

$$
\begin{aligned}
H(\mathbf{y}, \mathbf{p}) &= -\sum_k y_k \ln p_k \\
&= -\big[\, 0 \cdot \ln 0.2 + 0 \cdot \ln 0.5 + 1 \cdot \ln 0.3 \,\big] \\
&= -\ln 0.3 \approx 1.20 \text{ nats}
\end{aligned}
$$

The zeros in $\mathbf{y}$ wipe out every class except the observed one. So **the loss on one data point is $-\ln p_{\text{true}}$**: the network's surprise at the right answer, just like your surprise on one day of weather. It's this section's $-\ln q$, with $q = p_{\text{true}}$.

**From one point to a minibatch.**

- **Averaged over all $n$ training points,** it's the training loss: an estimate, from $n$ samples, of the cross-entropy we actually want.
- **Averaged over a minibatch,** it's a rougher, cheaper estimate, like the 10-day estimates in Demo S6. This is why [last week's minibatches](ch_6_and_10_03_training_networks.ipynb#section-7) gave noisy steps that still headed the right way. `MLPClassifier` uses minibatches of 200 by default.
- **The slopes come along.** The slope of an average is the average of the slopes. Backpropagation finds each point's slopes from its own error signal and its own inputs, and the minibatch's slopes are their average: an estimate of the slopes of the cross-entropy we want. Every gradient descent step uses that estimate.

**Single labels can still teach probabilities.** If similar sloppy 4s are labeled "4" seven times out of ten and "9" three times, the network makes its average surprise smallest by predicting about 70% four and 30% nine, even though no single label says that.

**The "knows nothing" loss.** A network that hasn't learned anything spreads its probability evenly over the $K$ classes, so $p_{\text{true}} = 1/K$ and the loss is $\ln K$. For 10 digits, that's $\ln 10 \approx 2.30$. Lab 7's networks started near it (the 2-unit network's loss after its first epoch was 2.33), and training has to get below it.

<details>
<summary>Optional: what units is this loss in?</summary>

Natural logs measure surprise in **nats**, just as $\log_2$ measures it in bits: $\ln 10 \approx 2.30$ nats is the same surprise as $\log_2 10 \approx 3.32$ bits (1 nat ≈ 1.44 bits). Libraries report losses as plain numbers, so Lab 7's 2.33 is in nats.

</details>

**Ask:** Next week, we'll see a language model that predicts the next token from a vocabulary of about 50,000 tokens. Near what cross-entropy loss do you expect its training to start? 


<details>
<summary>Answer</summary>

**$\ln 50{,}000 \approx 10.8$.** Before training, the model knows nothing, so it spreads its probability about evenly over the 50,000 tokens: $p_{\text{true}} \approx 1/50{,}000$, and the loss is $-\ln \tfrac{1}{50{,}000} = \ln 50{,}000$. It's the same "knows nothing" rule as $\ln 10$ for the digits, with $K = 50{,}000$.

</details>

<a id="summary"></a>

## Summary

- **Softmax** turns $K$ scores into $K$ probabilities: raise $e$ to each score, then divide by the total. Only differences between scores matter.
- **Surprise** is $-\log_2 q(x)$ bits: rare outcomes are surprising, certain ones aren't, and independent surprises add.
- **Shannon entropy** $H(p)$ is the average surprise when you know the truth: the average number of bits the best code needs. The uniform distribution has the most; a certain outcome has none.
- **Cross-entropy** $H(p, q)$ is your average surprise when the truth is $p$ but you believe $q$. The average is over the truth; the surprise is measured by your beliefs.
- **KL divergence** $D_{\mathrm{KL}}(p \,\Vert\, q) = H(p, q) - H(p)$ is the extra surprise you pay for believing $q$. It's never negative, it's 0 only when $q = p$, and it isn't symmetric.
- **Cross-entropy beats squared error as a training loss:** it punishes confident mistakes without limit, and its slope with respect to the scores is largest when the model is most wrong, so gradient descent takes its biggest steps there.
- **We never see the truth, only samples of it,** so we estimate the cross-entropy by averaging surprise over samples. One labeled data point is one sample: its one-hot label $\mathbf{y}$ leaves a loss of $-\ln p_{\text{true}}$, the network's surprise at the right answer. The training loss averages this over the training data; a minibatch is a rougher, cheaper estimate, and so are its slopes.
- **A $K$-class model that knows nothing** scores $\ln K$: about 2.30 for 10 digits.

> **A note on symbols, going forward.** Through most of Section 5, $p$ was the truth and $q$ your beliefs: the standard letters in information theory. From the end of Section 5 on, and in the next notebook, we're back to the usual machine-learning convention from Lab 7 and last week: $\mathbf{p}$ is the **network's predicted probabilities**, and the truth is the label $\mathbf{y}$.

Every large language model is trained to make cross-entropy small, over a vocabulary of tens of thousands of tokens instead of 10 digits. That's next week.

<a id="section-6"></a>

## 6 · Back to the network: you've been using cross-entropy all along

It turns out you've been using cross-entropy all along: last week's log-loss *is* the cross-entropy from Section 4. [The end of Section 5](#loss-one-point) gave the short version, $-\ln p_{\text{true}}$; this section derives it step by step. We'll start from Section 4's definition, plug in what the truth and the model are for one labeled data point, simplify, and arrive at the log-loss you already know ([last week, Section 2](ch_6_and_10_03_training_networks.ipynb#section-2)), where it came from the likelihood.

### Optional: The rest of the notebook up until the Practice Questions is optional. The Practice Questions, are *not* optional.

> **A note on symbols.** Steps 1 to 3 use the information-theory letters ($p$ the truth, $q$ the beliefs). Steps 4 and 5 plug in the label and the network's prediction, and from then on $p$ means "the network's predicted probability" again, as at the end of Section 5.

**The two-class case, one step at a time.** Take one data point $i$, with label $y_i \in \{0, 1\}$ and the network's prediction $p_i$, its probability of class 1, exactly as last week.

**Step 1. Start from the definition** in [Section 4](#section-4):

$$
H(p, q) = -\sum_x p(x)\,\log_2 q(x)
$$

**Step 2. Switch to natural logs.** $\ln q = (\ln 2)\log_2 q$, so this only multiplies every value by the constant $\ln 2 \approx 0.69$: the units change from bits to **nats**, and nothing else does. Networks, scikit-learn, and ISL (eq. 10.14) use $\ln$:

$$
H(p, q) = -\sum_x p(x)\,\ln q(x)
$$

**Step 3. List the outcomes.** There are two: class 1 and class 0. Write the sum as its two terms:

$$
H(p, q) = -\big[\, p(1)\ln q(1) + p(0)\ln q(0) \,\big]
$$

**Step 4. The truth for one labeled data point.** We *know* its class, so the true distribution puts all its probability on the label:

| Label | $p(1)$ | $p(0)$ |
|---|---|---|
| $y_i = 1$ | 1 | 0 |
| $y_i = 0$ | 0 | 1 |

One formula covers both rows:

$$
p(1) = y_i, \qquad p(0) = 1 - y_i
$$

(Check: plug in $y_i = 1$, then $y_i = 0$, and you get each row back.)

**Step 5. The model's distribution.** The network's output $p_i$ is its probability of class 1, and the two probabilities sum to 1:

$$
q(1) = p_i, \qquad q(0) = 1 - p_i
$$

**Step 6. Substitute** Steps 4 and 5 into Step 3:

$$
\begin{aligned}
H(p, q) &= -\big[\, p(1)\ln q(1) + p(0)\ln q(0) \,\big] \\
&= -\big[\, y_i \ln p_i + (1 - y_i)\ln(1 - p_i) \,\big]
\end{aligned}
$$

That bracket is exactly each data point's term in last week's log-loss.

**Step 7. Average over the $n$ training points.** Each data point has its own truth (its label), so compute each one's cross-entropy and average:

$$
L = -\frac{1}{n}\sum_{i=1}^{n} \Big[\, y_i \ln p_i + (1 - y_i)\ln(1 - p_i) \Big]
$$

**This is last week's log-loss, symbol for symbol.** You've been using cross-entropy all along.

**The KL view agrees.** A labeled data point's truth is certain, so its Shannon entropy is 0: no surprise when you already know the answer. So here the cross-entropy and the KL divergence are equal, and minimizing the log-loss is literally moving the model's beliefs toward the truth.

### Ten classes

The same steps, with $K$ classes instead of 2:

- **The truth** is a **one-hot** vector $\mathbf{y}$: 1 in the true class's entry, 0 everywhere else. For an image of a 3, with classes $0, 1, \dots, 9$: $\mathbf{y} = (0, 0, 0, 1, 0, 0, 0, 0, 0, 0)$. That's Step 4 with $K$ entries instead of 2.
- **The model** is the softmax output $\mathbf{p} = (p_1, \dots, p_K)$ from [Section 1](#section-1).
- **Substitute and simplify.** Every term with $y_k = 0$ vanishes, leaving one:

$$
\begin{aligned}
H(\mathbf{y}, \mathbf{p}) &= -\sum_{k=1}^{K} y_k \ln p_k \\
&= -\big[\, 0 \cdot \ln p_1 + \cdots + 1 \cdot \ln p_{\text{true}} + \cdots + 0 \cdot \ln p_K \,\big] \\
&= -\ln p_{\text{true}}
\end{aligned}
$$

**The loss on one image is minus the log of the probability the network gave the right answer.** The two-class case says the same thing: last week's two cases were $-\ln p_i$ when $y_i = 1$ and $-\ln(1 - p_i)$ when $y_i = 0$. Averaged over the training images, this is the loss `MLPClassifier` minimized in Lab 7.

**The "knows nothing" loss.** A network that gives all 10 digits probability $1/10$ has loss $-\ln \tfrac{1}{10} = \ln 10 \approx 2.30$ on every image: the 10-class version of last week's $\ln 2$. Lab 7's networks started near this value: the 2-unit network's loss after its first epoch was 2.33. Look for it at the start of your own loss curves.

**Code worth reading.** The cross-entropy loss for a batch of images, from the scores. `Y` is one-hot, so `Y * np.log(P)` keeps only each row's true-class entry.

In [ ]:
def cross_entropy(Z, Y):
    "Average cross-entropy, in nats, of scores Z (n, K) against one-hot labels Y (n, K)."
    P = softmax(Z)
    return -np.mean(np.sum(Y * np.log(P), axis=1))


Y = np.array([[1, 0, 0],                 # data point 1 is class 1
              [0, 0, 1]])                # data point 2 is class 3
print('loss:', round(cross_entropy(Z, Y), 3))
print('same, as -ln(p_true):', round(-np.mean(np.log([P[0, 0], P[1, 2]])), 3))
print('knows nothing, 10 classes:', round(cross_entropy(np.zeros((1, 10)), np.eye(10)[[3]]), 3),
      ' ln 10 =', round(np.log(10), 3))

### The error signal is $\mathbf{p} - \mathbf{y}$ again

Backpropagation starts from the output's error signal, the slope of the loss with respect to each output score ([last week, Step 4](ch_6_and_10_03_training_networks.ipynb#section-5)). With a sigmoid output and log-loss it was $p - y$. With softmax and cross-entropy it's the same pattern, one entry per class:

$$
\frac{\partial L}{\partial z_k} = p_k - y_k \qquad \text{for } k = 1, \dots, K
$$

Prediction minus target. For the true class, $p_k - 1$ is negative, so gradient descent raises that score; for every other class, $p_k - 0$ is positive, so it lowers them.

<details>
<summary>Optional: deriving the error signal</summary>

Take the log of softmax:

$$
\begin{aligned}
\ln p_j &= \ln e^{z_j} - \ln \sum_{m} e^{z_m} \\
&= z_j - \ln \sum_{m} e^{z_m}
\end{aligned}
$$

Substitute into the loss for one data point, and use $\sum_j y_j = 1$ (one-hot):

$$
\begin{aligned}
L &= -\sum_j y_j \ln p_j \\
&= -\sum_j y_j z_j + \Big(\sum_j y_j\Big) \ln \sum_m e^{z_m} \\
&= -\sum_j y_j z_j + \ln \sum_m e^{z_m}
\end{aligned}
$$

Differentiate with respect to $z_k$. The first sum has one term with $z_k$ in it, and the second uses the chain rule:

$$
\begin{aligned}
\frac{\partial L}{\partial z_k} &= -y_k + \frac{e^{z_k}}{\sum_m e^{z_m}} \\
&= p_k - y_k
\end{aligned}
$$

</details>

**Code worth reading.** The check, as last week: nudge each score both ways (finite differences, [last week's Section 4](ch_6_and_10_03_training_networks.ipynb#section-4)) and compare with $\mathbf{p} - \mathbf{y}$. One data point, three classes.

In [ ]:
z = np.array([[2.0, 1.0, 0.0]])          # one data point's three scores
y = np.array([[0, 1, 0]])                # its true class is class 2
h = 1e-5

numerical = np.zeros_like(z)
for k in range(z.shape[1]):              # nudge each score up and down
    up, down = z.copy(), z.copy()
    up[0, k] += h
    down[0, k] -= h
    numerical[0, k] = (cross_entropy(up, y) - cross_entropy(down, y)) / (2 * h)

formula = softmax(z) - y
print('finite differences:', numerical)
print('p - y:             ', formula)
print(f'largest difference: {np.abs(numerical - formula).max():.1e}')

They agree. So every output layer we've met starts backpropagation the same way:

| Task | Output activation | Loss | Output error signal |
|---|---|---|---|
| Regression | none | squared error, $\tfrac12(\hat{y} - y)^2$ | $\hat{y} - y$ |
| Two classes | sigmoid | log-loss (two-class cross-entropy) | $p - y$ |
| $K$ classes | softmax | cross-entropy | $\mathbf{p} - \mathbf{y}$, one entry per class |

These pairings aren't automatic: [Section 5](#section-5) showed that a sigmoid scored with squared error gives an error signal with an extra factor that vanishes when the network is confidently wrong. Each loss is paired with the output activation that makes the error signal come out as prediction minus target.

**Ask:** A 10-class network gives the true digit probability 0.9. What's its loss on that image? What if it gives the true digit 0.01?


<details>
<summary>Answer</summary>

The loss is $-\ln p_{\text{true}}$:

- $-\ln 0.9 \approx 0.105$: confident and right, a small loss.
- $-\ln 0.01 \approx 4.6$: about twice the "knows nothing" loss of $\ln 10 \approx 2.30$. Confidently wrong is worse than knowing nothing.

</details>

<a id="optional-summary"></a>

## Summary of the optional material

- **The sigmoid is softmax with two classes** ([Section 1](#section-1)'s optional box): for two scores, softmax gives $p_1 = \sigma(z_1 - z_2)$.
- **The KL divergence is never negative** ([Section 4](#section-4)'s optional proof): it follows from $\ln t \le t - 1$.
- **Last week's log-loss is cross-entropy:** it's the two-class case of $-\ln p_{\text{true}}$ ([Section 6](#section-6), step by step).
- **Softmax with cross-entropy** starts backpropagation with $\mathbf{p} - \mathbf{y}$, the same pattern as last week.

# Practice Questions

These questions cover softmax, surprise, Shannon entropy, cross-entropy and KL divergence, why cross-entropy is the training loss, and cross-entropy as a network's loss. Questions like these will appear on the learning check, so try each one on paper before you open its hint or answer.

For reference:

- **The weather truth** (Sections 3 and 4): $p$ = (sunny $1/2$, cloudy $1/4$, rain $1/8$, snow $1/8$), with $H(p) = 1.75$ bits.
- **Softmax:** $p_k = \dfrac{e^{z_k}}{\sum_{j} e^{z_j}}$.
- **Surprise:** $-\log_2 q(x)$ bits. **Shannon entropy:** $H(p) = -\sum_x p(x)\log_2 p(x)$. **Cross-entropy:** $H(p, q) = -\sum_x p(x)\log_2 q(x)$. **KL divergence:** $D_{\mathrm{KL}}(p \,\Vert\, q) = H(p, q) - H(p)$.
- **Logs you may need:** $\log_2 2 = 1$, $\log_2 4 = 2$, $\log_2 8 = 3$, $\log_2 16 = 4$, $\log_2 32 = 5$; $\ln 2 \approx 0.693$, $\ln 10 \approx 2.303$.

### Question 1: What the output layer needs

A network classifies images into 10 classes. Its output layer computes 10 scores, $z_1, \dots, z_{10}$.

(a) Select **all** the properties the 10 outputs (one per output neuron) must have to be probabilities:

- A. Each is between 0 and 1.
- B. They can be negative, like the scores.
- C. Together they sum to 1.
- D. A bigger score gives a bigger probability.
- E. Exactly one of them equals 1.

(b) Why doesn't putting a sigmoid on each output unit, as last week's network did with its one output, work here?

(See [The output layer of a multi-class network](#output-layer).)

<details>
<summary>Answer</summary>

**(a) A, C, and D.**

- B is wrong: probabilities can't be negative, which is exactly what's wrong with raw scores.
- E is wrong: an output of exactly 1 would mean the network is certain. Softmax gives every class *some* probability.

**(b) The ten sigmoid outputs don't have to sum to 1.** Each sigmoid answers its own yes/no question ("is it a 3?"), so the network could say 0.9 to both "3" and "8". Softmax fixes all three properties at once, and it replaces the sigmoid on the output layer.

</details>

### Question 2: Softmax by hand

A network has three output scores. Raising $e$ to each score gives

| Class | 1 | 2 | 3 |
|---|---|---|---|
| $e^{z_k}$ | 6 | 3 | 1 |

(a) What are the three softmax probabilities?

(b) Now add 2 to every score. What happens to the probabilities?

(See [Section 1](#section-1).)

<details>
<summary>Hint</summary>

Softmax has two steps, and the first one ($e$ to each score) is already done for you.

</details>

<details>
<summary>Answer</summary>

**(a) 0.6, 0.3, and 0.1.** The total is $6 + 3 + 1 = 10$, and each probability is its $e^{z_k}$ divided by the total: $6/10$, $3/10$, $1/10$. They sum to 1.

**(b) Nothing changes.** Every $e^{z_k}$ is multiplied by the same factor $e^2$, which cancels between the top and the bottom of softmax. Only the differences between the scores matter.

</details>

### Question 3: Surprise

(a) You gave an outcome the probability $1/32$, and that outcome happened. How many bits of surprise is that?

(b) Two independent events each had probability $1/4$, and both happened. What's your total surprise? Compute it two ways: from the probability that both happen, and as a sum of the individual surprises.

(c) Which surprises you more: an outcome you gave probability 0.9, or one you gave probability 0.1?

(See [Section 2](#section-2).)

<details>
<summary>Answer</summary>

**(a) 5 bits of surprise.** $-\log_2 \tfrac{1}{32} = \log_2 32 = 5$.

**(b) 4 bits of total surprise, both ways.**

- Both happen with probability $\tfrac14 \cdot \tfrac14 = \tfrac{1}{16}$, and $-\log_2 \tfrac{1}{16} = 4$.
- Each event alone is $-\log_2 \tfrac14 = 2$ bits, and $2 + 2 = 4$.

The log turns the product of the probabilities into a sum of the surprises.

**(c) The outcome you gave 0.1.** The less likely you thought it was, the more surprising it is: $-\log_2 0.1 \approx 3.3$ bits, against $-\log_2 0.9 \approx 0.15$ bits.

</details>

### Question 4: Shannon entropy by hand

A distribution over three outcomes, with each outcome's surprise already computed:

| Outcome | A | B | C |
|---|---|---|---|
| $p(x)$ | $1/2$ | $1/4$ | $1/4$ |
| surprise $-\log_2 p(x)$ | 1 | 2 | 2 |

What is its Shannon entropy $H(p)$, in bits?

(See [Section 3](#section-3).)

<details>
<summary>Hint</summary>

Multiply each surprise by how often that outcome happens, then add.

</details>

<details>
<summary>Answer</summary>

**1.5 bits.** Shannon entropy is the average surprise, weighted by how often each outcome happens:

$$
\begin{aligned}
H(p) &= \tfrac12 \cdot 1 + \tfrac14 \cdot 2 + \tfrac14 \cdot 2 \\
&= 0.5 + 0.5 + 0.5 \\
&= 1.5 \text{ bits}
\end{aligned}
$$

A common mistake is to add the surprises without weighting them (giving 5), or to average them equally (giving $5/3$).

</details>

### Question 5: More or less entropy

(a) Rank these three distributions (X, Y and Z are the distributions) over the same 4 outcomes from the **least** Shannon entropy to the **most**:

- X: $(1/4, 1/4, 1/4, 1/4)$
- Y: $(1/2, 1/4, 1/8, 1/8)$ (the weather)
- Z: $(1, 0, 0, 0)$

(b) True or false: Shannon entropy is a property of a single outcome.

(See [Section 3](#section-3).)

<details>
<summary>Answer</summary>

**(a) Z (0 bits), then Y (1.75 bits), then X (2 bits).**

- Z is certain, so nothing ever surprises you: the least.
- The uniform distribution X has the most: every outcome is equally unpredictable.

**(b) False.** Shannon entropy belongs to the whole **distribution**: it averages the surprise over every outcome. Surprise is the property of a single outcome.

</details>

### Question 6: Reading the cross-entropy plot (I)

The weather really follows $p$ = (sunny $1/2$, cloudy $1/4$, rain $1/8$, snow $1/8$), but your beliefs are $q$ = (sunny $1/4$, cloudy $1/2$, rain $1/8$, snow $1/8$): you swapped sunny and cloudy. The figure below is Demo S4 at this setting, with the numbers on the "believe q" bar hidden.

(a) Compute sunny's piece and cloudy's piece of the cross-entropy. Each piece is $p(x) \cdot \big(-\log_2 q(x)\big)$.

(b) Rain's and snow's pieces are 0.375 bits each, the same as in the "believe p" bar, because you got their probabilities right. Find the cross-entropy $H(p, q)$ and the KL divergence.

(c) Your belief about cloudy made cloudy's piece *smaller* than its piece in the "believe p" bar. So why is the total, $H(p, q)$, still bigger than $H(p)$?

(See [Section 4](#section-4).)

In [ ]:
from ch_6_and_10_demos_helpers_2 import plot_cross_entropy_snapshot

plot_cross_entropy_snapshot([1/4, 1/2, 1/8, 1/8]);

<details>
<summary>Hint</summary>

How often an outcome happens comes from the truth $p$; how surprised you are when it does comes from your beliefs $q$. For (c), compare how much sunny's piece grew with how much cloudy's shrank.

</details>

<details>
<summary>Answer</summary>

**(a) Sunny: 1 bit. Cloudy: 0.25 bits.**

$$
\begin{aligned}
\text{sunny: } \tfrac12 \cdot \big(-\log_2 \tfrac14\big) &= \tfrac12 \cdot 2 = 1 \\
\text{cloudy: } \tfrac14 \cdot \big(-\log_2 \tfrac12\big) &= \tfrac14 \cdot 1 = 0.25
\end{aligned}
$$

**(b) $H(p, q) = 2$ bits, and the KL divergence is 0.25 bits.**

$$
\begin{aligned}
H(p, q) &= 1 + 0.25 + 0.375 + 0.375 = 2 \\
D_{\mathrm{KL}}(p \,\Vert\, q) &= H(p, q) - H(p) = 2 - 1.75 = 0.25
\end{aligned}
$$

**(c) Because sunny's piece grew by more than cloudy's shrank.** The extra probability you gave cloudy had to come from somewhere: here, from sunny. Cloudy's piece dropped by 0.25 bits (from 0.5 to 0.25), but sunny's grew by 0.5 bits (from 0.5 to 1), because sunny happens twice as often as cloudy and you now underrate it. Moving probability away from the truth always costs more on the outcomes you underrate than it saves on the ones you overrate, so $H(p, q)$ is never below $H(p)$.

</details>

### Question 7: Reading the cross-entropy plot (II)

Same truth $p$ as Question 6, but now your beliefs are reversed: $q$ = (sunny $1/8$, cloudy $1/8$, rain $1/4$, snow $1/2$). The numbers on the "believe q" bar are hidden again.

(a) Compute all four pieces of the cross-entropy, and $H(p, q)$.

(b) Which outcome's piece is the smallest? Explain why, using how often it happens and how surprised you are when it does.

(c) Find the KL divergence. Which beliefs are closer to the truth, Question 6's or these?

(See [Section 4](#section-4).)

In [ ]:
from ch_6_and_10_demos_helpers_2 import plot_cross_entropy_snapshot

plot_cross_entropy_snapshot([1/8, 1/8, 1/4, 1/2]);

<details>
<summary>Answer</summary>

**(a) Sunny 1.5, cloudy 0.75, rain 0.25, snow 0.125, so $H(p, q) = 2.625$ bits.**

$$
\begin{aligned}
\text{sunny: } \tfrac12 \cdot 3 &= 1.5 \\
\text{cloudy: } \tfrac14 \cdot 3 &= 0.75 \\
\text{rain: } \tfrac18 \cdot 2 &= 0.25 \\
\text{snow: } \tfrac18 \cdot 1 &= 0.125 \\
H(p, q) &= 1.5 + 0.75 + 0.25 + 0.125 = 2.625
\end{aligned}
$$

**(b) Snow, at 0.125 bits.** Snow is rare ($p = 1/8$), and you expected it ($q = 1/2$), so a snowy day surprises you by only 1 bit, and it comes only one day in eight.

**(c) 0.875 bits, so Question 6's beliefs are closer.** $2.625 - 1.75 = 0.875$ bits, against Question 6's 0.25 bits. A smaller KL divergence means beliefs closer to the truth: they cost less extra surprise per day.

</details>

### Question 8: What did the model believe?

The figure shows the weather truth $p$ and the two stacked bars for some beliefs $q$, but not $q$ itself, and the "believe q" bar's numbers are hidden. Answer each part by comparing the pictures, without computing anything.

(a) Sunny's piece and rain's piece are the same size in both bars. What does that tell you about the model's beliefs about sunny and rain?

(b) Cloudy's piece is *bigger* in the "believe q" bar. Did the model believe cloudy was more likely or less likely than the actual probability, $p(\text{Cloudy})$? Snow's piece is *smaller*. What about snow?

(c) To shrink the hatched KL strip, which way should the model move probability: from snow to cloudy, or from cloudy to snow?

(See [Section 4](#section-4).)

In [ ]:
from ch_6_and_10_demos_helpers_2 import plot_cross_entropy_snapshot

plot_cross_entropy_snapshot([1/2, 1/8, 1/8, 1/4], mystery=True);

<details>
<summary>Hint</summary>

Each piece is (how often the outcome happens, from the truth $p$) $\times$ (how surprised you are when it does, from the beliefs $q$). The first factor is the same in both bars, so any change in a piece's size comes from the beliefs.

</details>

<details>
<summary>Answer</summary>

**(a) The model got them exactly right: $q = p$ for sunny and for rain.** In both bars, a piece's "how often" factor comes from the same truth $p$. So the same size means the same surprise, which means the same belief.

**(b) Cloudy: less likely than it really is (underrated). Snow: more likely than it really is (overrated).** A bigger piece means each cloudy day surprised the model more than it should have, so the model gave cloudy too little probability. A smaller piece means snowy days surprised it less, so it gave snow too much.

**(c) From snow to cloudy.** That moves both beliefs toward the truth: cloudy up from underrated, snow down from overrated. Shrinking the KL divergence by moving beliefs toward the truth is exactly what training does.

</details>

### Question 9: KL divergence, true or false

Mark each statement true or false.

(a) The KL divergence $D_{\mathrm{KL}}(p \,\Vert\, q)$ can be negative if your beliefs $q$ happen to favor the right outcomes.

(b) The KL divergence is 0 only when $q = p$.

(c) $D_{\mathrm{KL}}(p \,\Vert\, q) = D_{\mathrm{KL}}(q \,\Vert\, p)$ for every pair of distributions.

(d) For a fixed truth $p$, making the cross-entropy $H(p, q)$ smaller is the same as making the KL divergence smaller.

(e) In the cross-entropy $H(p, q)$, the average is taken over your beliefs $q$.

(See [Section 4](#section-4).)

<details>
<summary>Answer</summary>

**(a) False.** The KL divergence is never negative: no beliefs beat the truth on average.

**(b) True.**

**(c) False.** The KL divergence isn't symmetric. For a 90% coin and a fair coin, the two directions give about 0.53 and 0.74 bits.

**(d) True.** $D_{\mathrm{KL}}(p \,\Vert\, q) = H(p, q) - H(p)$, and $H(p)$ doesn't depend on $q$. So lowering one lowers the other by the same amount.

**(e) False.** The average is over the **truth** $p$: how often each outcome really happens. Your beliefs $q$ only set the surprise.

</details>

### Question 10: Why not squared error?

A model gives probability $q$ to what actually happened, through a sigmoid output $q = \sigma(z)$. The table gives each loss and its slope with respect to the score, $\partial L / \partial z$. Gradient descent changes $z$ by $-\eta\, \partial L / \partial z$, so the slope's size sets how big a step training takes on $z$ (informally, how hard the loss "pushes" on $z$).

| $q$ | Squared error $(1 - q)^2$ | Cross-entropy $-\ln q$ | Slope $\partial L / \partial z$, squared error | Slope $\partial L / \partial z$, cross-entropy |
|---|---|---|---|---|
| 0.5 | 0.25 | 0.69 | −0.25 | −0.50 |
| 0.01 | 0.98 | 4.61 | −0.020 | −0.99 |

Select **the two** statements that explain why networks are trained with cross-entropy rather than squared error:

- A. Squared error can never exceed 1, so it barely distinguishes a confident mistake from a mild one; cross-entropy keeps growing.
- B. Cross-entropy's slope $\partial L / \partial z$ is largest in size when the model is most wrong; squared error's slope nearly vanishes there, so gradient descent barely changes $z$.
- C. Squared error can't be computed for probabilities.
- D. Cross-entropy is always smaller than squared error, so training finishes sooner.

(See [Section 5](#section-5).)

<details>
<summary>Answer</summary>

**A and B.**

- **A:** going from $q = 0.5$ to $q = 0.01$ moves squared error only from 0.25 to 0.98, but cross-entropy from 0.69 to 4.61.
- **B:** at $q = 0.01$, the model is confidently wrong, and squared error's slope is only −0.020, against −0.99 for cross-entropy. Gradient descent's step is proportional to the slope, so a network trained with squared error barely moves exactly when it most needs to.
- **C is wrong:** squared error on probabilities can be computed, and it's even a reasonable way to *evaluate* a model (the Brier score). Its problems are as a *training* loss.
- **D is wrong:** at $q = 0.5$, cross-entropy (0.69) is bigger than squared error (0.25). And a smaller loss value doesn't mean faster training.

</details>

### Question 11: The loss for one data point

(a) A 10-class network gives the true digit probability 0.5. What's its cross-entropy loss on that image?

(b) After 50 epochs, a 10-class network's training loss is still about 2.3. What does that tell you?

(See [The loss for one labeled data point](#loss-one-point).)

<details>
<summary>Hint</summary>

Every part uses the same rule: the loss on one data point is minus the natural log of the probability the model gave the true class.

</details>

<details>
<summary>Answer</summary>

**(a) $\ln 2 \approx 0.693$.** For a labeled data point, the cross-entropy is $-\ln p_{\text{true}} = -\ln 0.5 = \ln 2$.

**(b) It has learned almost nothing.** $\ln 10 \approx 2.30$ is the loss of a network that gives all 10 classes probability $1/10$. A network that's learning should get well below it.

</details>

### Question 12: Training without the truth

(a) Cross-entropy averages surprise over the truth, but we never know the true distribution behind our data. How can we still use cross-entropy as a training loss?

(b) The average loss over one minibatch is an estimate of which of these?

- A. The true distribution of the labels.
- B. The network's predicted probabilities.
- C. The cross-entropy between the truth and the network's beliefs.
- D. The number of classes.

(c) Two networks are trained the same way, one with minibatches of 10 and one with minibatches of 200. Whose minibatch losses vary more from step to step? Why does training still work with the noisier one?

(See [Training on samples](#loss-one-point).)

<details>
<summary>Hint</summary>

Think of the weather record in Demo S6: you never used $p$ to compute the estimate, only the days it produced.

</details>

<details>
<summary>Answer</summary>

**(a) By averaging over samples.** The training data are samples from the truth. Averaging the network's surprise, $-\ln p_{\text{true}}$, over them estimates the cross-entropy, just as averaging your surprise over a year of recorded weather estimates $H(p, q)$ without knowing $p$.

**(b) C.** The network's predicted probabilities are known exactly (it computes them), and the true distribution is never estimated on its own: the samples just stand in for it in the average.

**(c) The minibatches of 10.** An average over fewer samples is noisier, like the 10-day estimates in Demo S6. But each estimate is centered on the right value, so on average the steps head the right way, and many cheap noisy steps still make progress.

</details>